# Clustering with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/clustering/01_clustering.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Task definition

Clustering discovers groups without using target labels. `TabFORGEClusterer` fits an unsupervised
TabFORGE embedding workflow and a lightweight KMeans head. Labels appear only in held-out evaluation.

In [ ]:
%pip install "tabforge-ml[tutorials]"

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    TabFORGEArchitectureConfig,
    TabFORGEClusterer,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "256"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")
MODEL_PATH = os.environ.get("TABFORGE_TUTORIAL_MODEL_PATH", "mock")
CHECKPOINT = None if MODEL_PATH == "mock" else "pretrained"

## 2. Load and split with TabCamel

Vehicle has multiclass structure suitable for external cluster evaluation. The split and row cap make
the example reproducible and quick.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("vehicle", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set, test_set = split["train_set"], split["test_set"]
X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)
n_clusters = int(y_train.nunique())
print({"train": X_train.shape, "test": X_test.shape, "clusters": n_clusters})

## 3. Minimal API and fitting

The public clustering controls stay minimal: `n_clusters`, `random_state`, and common TabFORGE
configuration. `fit` sets `labels_`; `fit_predict` combines both operations.

In [ ]:
clusterer = TabFORGEClusterer(
    n_clusters=n_clusters,
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path=MODEL_PATH, n_folds=0 if MODEL_PATH == "mock" else 2),
    architecture_config=(
        TabFORGEArchitectureConfig(
            embedding_dimension=8 if MODEL_PATH == "mock" else None,
            decoder_layers=1,
            decoder_heads=2,
            decoder_ffn_factor=2,
            denoiser_layers=1,
            denoiser_heads=2,
            denoiser_ffn_factor=2,
        )
        if MODEL_PATH == "mock"
        else None
    ),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=TabFORGERuntimeConfig(device=DEVICE),
    random_state=RANDOM_STATE,
).fit(X_train, checkpoint=CHECKPOINT)
labels_train = clusterer.labels_
labels_test = clusterer.predict(X_test)

## 4. Evaluation

ARI, NMI, and V-measure compare discovered groups with hidden class labels and are invariant to
cluster-number permutations.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

print(
    {
        "ari": adjusted_rand_score(y_test, labels_test),
        "nmi": normalized_mutual_info_score(y_test, labels_test),
        "v_measure": v_measure_score(y_test, labels_test),
    }
)

## 5. sklearn interoperability and checkpoints

The estimator supports cloning, parameter inspection, `fit_predict`, prediction for new rows, and exact
checkpoint restoration of both TabFORGE and the KMeans head.

In [ ]:
from sklearn.base import clone

assert clone(clusterer).get_params()["n_clusters"] == n_clusters
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = clusterer.save_checkpoint(Path(directory))
    restored = TabFORGEClusterer.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_labels = restored.predict(X_test.iloc[:8])
print({"checkpoint": checkpoint_path.name, "restored_labels": restored_labels.tolist()})

## 6. Reproducibility notes

Keep the split, row curation, checkpoint revision, embedding choice, `n_clusters`, and random seed
fixed across methods. Repeat evaluation with several split seeds and report mean, standard deviation,
and dataset-level ranks.